<a href="https://colab.research.google.com/github/Yvanne-Toukam/NASA-PROJET/blob/main/AstraCare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%pip install -q pymupdf chromadb sentence-transformers beautifulsoup4 requests \
  "python-dateutil>=2.9,<3" \
  "oauthlib>=3.2,<4" \
  "opentelemetry-api>=1.39,<=1.42.1" \
  "opentelemetry-sdk>=1.39,<=1.42.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-common 0.66b2 requires opentelemetry-sdk~=1.45.2, but you have opentelemetry-sdk 1.42.1 which is incompatible.


In [13]:
# === CELLULE 2 : setup / chemins / GPU ===
from google.colab import drive
import os, re, requests, torch
from bs4 import BeautifulSoup

try:
    drive.mount('/content/drive', force_remount=True)
    BASE = "/content/drive/MyDrive/rag_nasa"
except Exception as e:
    print("Drive indisponible, stockage local à la place :", e)
    BASE = "/content/rag_nasa"

CORPUS = "/content/corpus"
INDEX = os.path.join(BASE, "index")

os.makedirs(CORPUS, exist_ok=True)
os.makedirs(INDEX, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dossier de travail :", BASE)
print("GPU disponible :", torch.cuda.is_available())
print("Device utilisé :", device)

Mounted at /content/drive
Dossier de travail : /content/drive/MyDrive/rag_nasa
GPU disponible : True
Device utilisé : cuda


In [14]:
# === CELLULE 3 : téléchargement des sources et sauvegarde dans /content/corpus ===
import os, re, requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (projet etudiant NASA Space Apps)"}

PAGES = [
    "https://www.nasa.gov/hrp/hazards/",
    "https://www.nasa.gov/hrp/risks/",
    "https://www.nasa.gov/hrp/ongoing-hrp-research/",
    "https://www.asc-csa.gc.ca/eng/news/articles/2026/2026-02-26-monitoring-radiation-astronaut-health.asp",
    "https://www.nasa.gov/reference/risk-of-cardiovascular-adaptations/",
    "https://www.nasa.gov/reference/risk-of-altered-immune-system-responses/",
    "https://www.nasa.gov/reference/risk-of-spaceflight-induced-bone-changes/",
    "https://www.nasa.gov/reference/risk-from-inadequate-sleep-and-irregular-schedules/",
    "https://www.nasa.gov/reference/risk-of-behavioral-conditions-and-psychiatric-disorders/",
    "https://www.nasa.gov/reference/risk-of-reduced-cardiorespiratory-and-musculoskeletal-fitness/",
    "https://www.nasa.gov/reference/risk-of-radiation-induced-cancers/",
    "https://www.nasa.gov/directorates/esdmd/hhp/immune-risk/",
    "https://ntrs.nasa.gov/citations/20150008177",
    "https://ntrs.nasa.gov/citations/20100036637",
    "https://www.nasa.gov/feature/glenn/2021/protecting-astronaut-health-with-the-prick-of-a-finger",
]

for url in PAGES:
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
    except Exception as e:
        print("ECHEC", url, "->", e)
        continue

    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "form", "noscript"]):
        tag.decompose()

    zone = soup.find("main") or soup.find("article") or soup.body
    if zone is None:
        print("Aucune zone principale trouvée pour :", url)
        continue

    texte = zone.get_text("\n", strip=True)
    n = len(texte.split())
    if n < 150:
        print("TROP COURT (ignoré) :", url, "-", n, "mots")
        continue

    nom = re.sub(r"[^a-z0-9]+", "_", url.lower().replace("https://", "")).strip("_")[:90] + ".txt"
    chemin = os.path.join(CORPUS, nom)

    with open(chemin, "w", encoding="utf-8") as f:
        f.write(f"SOURCE: {url}\n\n{texte}")

    print("OK", nom, "-", n, "mots")

OK www_nasa_gov_hrp_hazards.txt - 649 mots
OK www_nasa_gov_hrp_risks.txt - 945 mots
OK www_nasa_gov_hrp_ongoing_hrp_research.txt - 686 mots
ECHEC https://www.asc-csa.gc.ca/eng/news/articles/2026/2026-02-26-monitoring-radiation-astronaut-health.asp -> ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
OK www_nasa_gov_reference_risk_of_cardiovascular_adaptations.txt - 254 mots
OK www_nasa_gov_reference_risk_of_altered_immune_system_responses.txt - 295 mots
OK www_nasa_gov_reference_risk_of_spaceflight_induced_bone_changes.txt - 311 mots
OK www_nasa_gov_reference_risk_from_inadequate_sleep_and_irregular_schedules.txt - 282 mots
OK www_nasa_gov_reference_risk_of_behavioral_conditions_and_psychiatric_disorders.txt - 440 mots
OK www_nasa_gov_reference_risk_of_reduced_cardiorespiratory_and_musculoskeletal_fitness.txt - 376 mots
OK www_nasa_gov_reference_risk_of_radiation_induced_cancers.txt - 391 mots
OK www_nasa_gov_directorates_esdmd_hhp_immune_risk.txt - 387 mot

In [15]:
# === CELLULE 4 : fiches manuscrites complémentaires ===
FICHES = {
    "fiche_incidence_symptomes_iss.txt": (
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC5098747",
        "Summary written by the team from the published study Incidence of clinical symptoms during "
        "long-duration orbital spaceflight (Crucian et al., International Journal of General Medicine, 2016). "
        "The study examined electronic medical records of 46 long-duration International Space Station crew "
        "members, about 20.57 flight years in total. The incidence of reported clinical events related to "
        "immune dysregulation categories was 3.40 events per flight year. Skin rashes were the most reported "
        "event (1.12 per flight year), followed by upper respiratory symptoms (0.97 per flight year). During "
        "flight, 46 percent of crew members reported an event judged notable, and among the notable events "
        "40 percent were rashes or hypersensitivities. A confirmatory diagnosis is often impossible in flight "
        "because very little medical laboratory equipment is available on the station, so events are recorded "
        "as reported symptoms."
    ),
    "fiche_co2_symptomes_iss.txt": (
        "http://hdl.handle.net/2060/20140003857",
        "Summary written by the team from the NASA report JSC-CN-30047, The Relationship Between CO2 Levels "
        "and CO2 Related Symptoms Reported on the ISS. Medical Operations, Toxicology and the Lifetime "
        "Surveillance of Astronaut Health analyzed carbon dioxide levels aboard the ISS against symptoms "
        "reported by the crew in flight. The analysis used 1716 observations, after removing the first 7 days "
        "of flight to avoid confusion with space adaptation syndrome. Headache was observed in 46 cases, about "
        "3 percent. Carbon dioxide level, age at launch, time in flight and data source were significantly "
        "associated with headache. For each 1 mmHg increase in CO2, the odds of a crew member reporting a "
        "headache doubled. Vision changes were reported inconsistently and did not align with CO2 levels. "
        "The authors support efforts to lower CO2 exposure limits in spacecraft."
    ),
}

for nom, (url, texte) in FICHES.items():
    with open(os.path.join(CORPUS, nom), "w", encoding="utf-8") as f:
        f.write(f"SOURCE: {url}\n\n{texte}")
    print("OK", nom, "-", len(texte.split()), "mots")

OK fiche_incidence_symptomes_iss.txt - 132 mots
OK fiche_co2_symptomes_iss.txt - 134 mots


In [16]:
# === CELLULE 5 : découpage en passages ===
import pymupdf

def decouper(texte, taille=250, chevauchement=40):
    mots = texte.split()
    pas = taille - chevauchement
    if pas <= 0:
        pas = taille
    return [" ".join(mots[i:i + taille]) for i in range(0, len(mots), pas) if mots[i:i + taille]]

passages = []

for racine, _, fichiers in os.walk(CORPUS):
    for nom in sorted(fichiers):
        chemin = os.path.join(racine, nom)
        ext = os.path.splitext(nom)[1].lower()
        source = nom

        if ext == ".pdf":
            with pymupdf.open(chemin) as doc:
                pages = [(i + 1, p.get_text()) for i, p in enumerate(doc)]
        elif ext == ".txt":
            contenu = open(chemin, encoding="utf-8", errors="ignore").read()
            if contenu.startswith("SOURCE:"):
                source = contenu.split("\n", 1)[0].replace("SOURCE:", "").strip()
            pages = [(1, contenu)]
        else:
            continue

        avant = len(passages)
        for num, texte in pages:
            for morceau in decouper(texte):
                passages.append({"texte": morceau, "source": source, "page": num})

        n = len(passages) - avant
        print(f"{source[:70]} : {n} passages")

print("Total :", len(passages), "passages")

http://hdl.handle.net/2060/20140003857 : 1 passages
https://pmc.ncbi.nlm.nih.gov/articles/PMC5098747 : 1 passages
https://www.nasa.gov/directorates/esdmd/hhp/immune-risk/ : 2 passages
https://www.nasa.gov/feature/glenn/2021/protecting-astronaut-health-wi : 4 passages
https://www.nasa.gov/hrp/hazards/ : 4 passages
https://www.nasa.gov/hrp/ongoing-hrp-research/ : 4 passages
https://www.nasa.gov/hrp/risks/ : 5 passages
https://www.nasa.gov/reference/risk-from-inadequate-sleep-and-irregula : 2 passages
https://www.nasa.gov/reference/risk-of-altered-immune-system-responses : 2 passages
https://www.nasa.gov/reference/risk-of-behavioral-conditions-and-psych : 3 passages
https://www.nasa.gov/reference/risk-of-cardiovascular-adaptations/ : 2 passages
https://www.nasa.gov/reference/risk-of-radiation-induced-cancers/ : 2 passages
https://www.nasa.gov/reference/risk-of-reduced-cardiorespiratory-and-m : 2 passages
https://www.nasa.gov/reference/risk-of-spaceflight-induced-bone-change : 2 passages
T

In [17]:
# === CELLULE 6 : embeddings ===
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer("BAAI/bge-m3", device=device)

vecteurs = modele.encode(
    [p["texte"] for p in passages],
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(len(vecteurs), "vecteurs calculés")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

36 vecteurs calculés


In [18]:
import sys, os, subprocess

# Installation automatique et silencieuse de chromadb si non présente
try:
    import chromadb
except ImportError:
    print("Installation de chromadb en cours...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chromadb"])
    import chromadb

# === CELLULE 7 : index Chroma ===
# Sécurité : Recréation des chemins si non définis
if 'INDEX' not in globals():
    if 'BASE' not in globals():
        BASE = "/content/drive/MyDrive/rag_nasa" if os.path.exists("/content/drive") else "/content/rag_nasa"
    INDEX = os.path.join(BASE, "index")
    os.makedirs(INDEX, exist_ok=True)

# Sécurité : Vérification que les passages et vecteurs existent en mémoire
if 'passages' not in globals() or 'vecteurs' not in globals():
    raise NameError(
        "Les données 'passages' ou 'vecteurs' sont introuvables en mémoire.\n"
        "Veuillez d'abord exécuter la Cellule 5 (découpage en passages) et la Cellule 6 (génération des embeddings)."
    )

client = chromadb.PersistentClient(path=INDEX)
try:
    client.delete_collection("nasa")
except Exception:
    pass

col = client.create_collection("nasa", metadata={"hnsw:space": "cosine"})

for i in range(0, len(passages), 256):
    lot = passages[i:i + 256]
    col.add(
        ids=[str(i + j) for j in range(len(lot))],
        documents=[p["texte"] for p in lot],
        embeddings=vecteurs[i:i + 256].tolist(),
        metadatas=[{"source": p["source"], "page": p["page"]} for p in lot],
    )

print(col.count(), "passages indexés")

36 passages indexés


In [19]:
# === CELLULE 8 : recherche ===
def chercher(question, k=3):
    emb = modele.encode([question], normalize_embeddings=True, convert_to_numpy=True)
    res = col.query(query_embeddings=emb.tolist(), n_results=k)

    print("\nQUESTION :", question)
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  [{meta['source'][:70]}] distance={dist:.2f}")
        print("   ", doc[:200].replace("\n", " "), "...")

In [20]:
# === CELLULE 9 : tests ===
questions = [
    "Quels sont les cinq dangers du vol spatial habité ?",
    "Comment l'apesanteur affecte-t-elle le coeur et les vaisseaux sanguins ?",
    "Pourquoi les astronautes perdent-ils de la densité osseuse ?",
    "Quels sont les effets du manque de sommeil en mission ?",
    "Quelle part de la radiation sur l'ISS vient des neutrons ?",
    "Recette de gâteau au chocolat",
]

for q in questions:
    chercher(q)


QUESTION : Quels sont les cinq dangers du vol spatial habité ?
  [https://www.nasa.gov/hrp/hazards/] distance=0.32
    SOURCE: https://www.nasa.gov/hrp/hazards/ Human Spaceflight Hazards 5 Hazards of Human Spaceflight Astronauts encounter five hazards as they journey through space. Recognizing these hazards allows NAS ...
  [https://www.nasa.gov/hrp/ongoing-hrp-research/] distance=0.34
    HRP has identified five hazards of human spaceflight: space radiation, isolation and confinement, distance from Earth, gravity (or lack thereof), and hostile or closed environments. Learn About the Fi ...
  [https://www.nasa.gov/hrp/risks/] distance=0.36
    SOURCE: https://www.nasa.gov/hrp/risks/ Human Spaceflight Hazards Risks of Human Spaceflight NASA’s Human Research Program (HRP) seeks to understand specific risks associated with the hazards of human ...

QUESTION : Comment l'apesanteur affecte-t-elle le coeur et les vaisseaux sanguins ?
  [https://www.nasa.gov/reference/risk-of-cardiovascular-

In [21]:
%pip install -q transformers accelerate

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device_llm = "cuda" if torch.cuda.is_available() else "cpu"
dtype_llm = torch.float16 if device_llm == "cuda" else torch.float32

print("Chargement de", LLM_ID, "sur", device_llm, "…")

tokenizer_llm = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=dtype_llm
).to(device_llm)

llm.eval()
print("LLM prêt.")

Chargement de Qwen/Qwen2.5-1.5B-Instruct sur cuda …


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM prêt.


In [26]:
# Compatibilité avec les noms employés dans les cellules précédentes.
collection_rag = globals().get("collection", globals().get("col"))

if collection_rag is None:
    raise NameError("Index Chroma introuvable : exécute d'abord la cellule qui le crée.")

if "modele" not in globals():
    raise NameError("Modèle d'embeddings `modele` introuvable : exécute d'abord sa cellule.")

def repondre(question, k=5, seuil_distance=0.65):
    question = question.strip()
    if not question:
        print("Écris une question.")
        return

    # Recherche des passages les plus proches.
    vecteur_question = modele.encode(
        [question],
        normalize_embeddings=True
    ).tolist()

    resultats = collection_rag.query(
        query_embeddings=vecteur_question,
        n_results=min(k, collection_rag.count()),
        include=["documents", "metadatas", "distances"]
    )

    documents = resultats["documents"][0]
    metadonnees = resultats["metadatas"][0]
    distances = resultats["distances"][0]

    extraits = []
    for doc, metadata, distance in zip(documents, metadonnees, distances):
        if not doc or distance > seuil_distance:
            continue

        metadata = metadata or {}
        source = metadata.get("source", "Source non indiquée")
        extraits.append((source, doc))

    if not extraits:
        print("Je n'ai pas trouvé de source suffisamment pertinente pour répondre.")
        return

    contexte = "\n\n".join(
        f"[S{i}] Source : {source}\nExtrait : {texte}"
        for i, (source, texte) in enumerate(extraits, start=1)
    )

    systeme = """
Tu es AstraCare, un assistant pédagogique sur la santé humaine en vol spatial.
Réponds en français, clairement et de façon approfondie.

Utilise les extraits fournis comme base factuelle. N'invente pas de chiffres,
de résultats d'études ou de recommandations médicales. Cite les sources dans
le texte avec [S1], [S2], etc. Si les extraits ne permettent pas de répondre,
dis-le clairement. Ce contenu est informatif et ne remplace pas un professionnel
de santé ; ne pose pas de diagnostic et ne prescris pas de traitement.
"""

    messages = [
        {"role": "system", "content": systeme},
        {
            "role": "user",
            "content": (
                f"Question : {question}\n\n"
                f"Extraits récupérés :\n{contexte}\n\n"
                "Répond de manière approfondie en citant les extraits pertinents."
            )
        }
    ]

    # Correction : Génération du template de chat en texte puis tokenisation avec return_tensors
    texte_complet = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    entree = tokenizer_llm(
        texte_complet,
        return_tensors="pt"
    ).input_ids.to(device_llm)

    with torch.inference_mode():
        sortie = llm.generate(
            entree,
            max_new_tokens=600,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer_llm.eos_token_id
        )

    nouveaux_tokens = sortie[0][entree.shape[-1]:]
    texte_reponse = tokenizer_llm.decode(
        nouveaux_tokens,
        skip_special_tokens=True
    )

    print("### Réponse\n")
    print(texte_reponse)

    print("\n### Sources")
    for i, (source, _) in enumerate(extraits, start=1):
        print(f"[S{i}] {source}")

In [27]:
# Exécutez d'abord la cellule précédente (redwSDkkDoNc) pour enregistrer le correctif.
repondre("Comment la microgravité affecte-t-elle les os des astronautes ?")

### Réponse

La microgravité, caractérisée par une réduction significative du poids gravitationnel, impacte profondément l'ossification des os des astronautes lorsqu'ils sont exposés à ces conditions extrêmes. Les effets de cette situation peuvent varier considérablement entre individus, mais les données collectées depuis les expéditions ISS indiquent que l'effet moyen sur le taux de résistance et de capacité respiratoire peut atteindre jusqu'à 10-12% de leur niveau pré-expédition. De plus, il existe une variabilité importante dans les niveaux de force musculaire et de respiration cardiaque, avec entre 12-38% des membres de l'équipage perdant plus de 20% de leurs capacités physiques initiales.

Les impacts spécifiques sur les os impliquent une diminution progressive de la densité osseuse, ce qui augmente la probabilité de fractures et de maladies liées au vieillissement tels que l'ostéoporose. Ces changements ont été observés dans divers domaines physiologiques, notamment la force musc

In [28]:
%pip install -q gradio

In [29]:
import torch
import gradio as gr
from transformers import pipeline

# Whisper fonctionne en français et est exécuté dans le notebook.
device_asr = 0 if torch.cuda.is_available() else -1

transcripteur = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device_asr
)

print("Reconnaissance vocale prête.")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (4 outputs left out)

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Reconnaissance vocale prête.


In [35]:
# Compatibilité avec les noms employés dans les cellules précédentes.
collection_rag = globals().get("collection", globals().get("col"))

if collection_rag is None:
    raise NameError("Index Chroma introuvable. Exécute d'abord la cellule qui le crée.")

if "modele" not in globals():
    raise NameError("Modèle d'embeddings `modele` introuvable.")

if "llm" not in globals() or "tokenizer_llm" not in globals():
    raise NameError("LLM ou tokenizer introuvable. Exécute d'abord leur cellule de chargement.")

def generer_reponse(question, historique=None, k=5, seuil_distance=0.65):
    question = (question or "").strip()
    if not question:
        return "Écris ou dicte une question."

    vecteur = modele.encode(
        [question],
        normalize_embeddings=True
    ).tolist()

    resultats = collection_rag.query(
        query_embeddings=vecteur,
        n_results=min(k, collection_rag.count()),
        include=["documents", "metadatas", "distances"]
    )

    documents = resultats["documents"][0]
    metadonnees = resultats["metadatas"][0]
    distances = resultats["distances"][0]

    extraits = []
    sources_vues = set()

    for doc, metadata, distance in zip(documents, metadonnees, distances):
        if not doc or distance is None or distance > seuil_distance:
            continue

        metadata = metadata or {}
        source = metadata.get("source", "Source non indiquée")

        if source not in sources_vues:
            sources_vues.add(source)
            extraits.append((source, doc))

    if not extraits:
        return (
            "Je n’ai pas trouvé de source suffisamment pertinente dans le corpus "
            "pour répondre avec confiance. Pour tout symptôme nouveau ou "
            "préoccupant, contacte l’équipe médicale de la mission."
        )

    contexte = "\n\n".join(
        f"[S{i}] Source : {source}\nExtrait : {texte[:2500]}"
        for i, (source, texte) in enumerate(extraits, start=1)
    )

    systeme = """
Tu es AstraCare, un assistant pédagogique sur la santé en vol spatial.
Réponds en français, clairement et de façon approfondie.

Appuie les faits sur les extraits fournis et cite-les avec [S1], [S2], etc.
N'invente pas de chiffres, de seuils médicaux, de diagnostics ou de traitements.
Si les extraits ne suffisent pas, indique clairement cette limite.

Pour une question sur un symptôme personnel, donne seulement des informations
générales et recommande de suivre le protocole de bord et de contacter l'équipe
médicale de la mission. En cas de symptôme grave ou d'aggravation rapide,
recommande de prévenir immédiatement l'équipe médicale et de suivre le
protocole d'urgence. Ne décide jamais si une personne est apte à poursuivre
sa mission. Tes réponses ne remplacent pas un professionnel de santé.
"""

    messages = [{"role": "system", "content": systeme}]

    # Garde un peu de contexte de conversation, sans envoyer tout l'historique.
    for message in (historique or [])[-6:]:
        if isinstance(message, dict) and message.get("role") in ("user", "assistant"):
            contenu = message.get("content", "")
            if isinstance(contenu, str) and contenu.strip():
                messages.append({
                    "role": message["role"],
                    "content": contenu
                })

    messages.append({
        "role": "user",
        "content": (
            f"Question : {question}\n\n"
            f"Extraits du corpus :\n{contexte}\n\n"
            "Réponds de manière utile et approfondie. Cite les sources dans "
            "ta réponse avec les marqueurs [S1], [S2], etc."
        )
    })

    # Correction : Génération sûre du template de chat puis tokenisation
    texte_complet = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    entree = tokenizer_llm(
        texte_complet,
        return_tensors="pt"
    ).to(next(llm.parameters()).device)

    with torch.inference_mode():
        sortie = llm.generate(
            input_ids=entree.input_ids,
            attention_mask=entree.attention_mask,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer_llm.eos_token_id
        )

    longueur_prompt = entree.input_ids.shape[-1]
    nouveaux_tokens = sortie[0][longueur_prompt:]
    texte = tokenizer_llm.decode(
        nouveaux_tokens,
        skip_special_tokens=True
    )

    liste_sources = "\n".join(
        f"[S{i}] {source}"
        for i, (source, _) in enumerate(extraits, start=1)
    )

    return f"{texte}\n\n**Sources consultées :**\n{liste_sources}"

In [ ]:
    if not fichier_audio:
        return ""

    try:
        resultat = transcripteur(
            fichier_audio,
            generate_kwargs={"task": "transcribe"}
        )
        return resultat.get("text", "").strip()
    except Exception as e:
        return f"Erreur de transcription : {e}"


def envoyer_message(message, historique):
    historique = historique or []

    # Convertit un ancien historique au format (question, réponse), si présent.
    historique_messages = []
    for element in historique:
        if isinstance(element, dict) and "role" in element and "content" in element:
            historique_messages.append(element)
        elif isinstance(element, (list, tuple)) and len(element) == 2:
            question, reponse = element
            if question:
                historique_messages.append({"role": "user", "content": str(question)})
            if reponse:
                historique_messages.append({"role": "assistant", "content": str(reponse)})

    message = (message or "").strip()
    if not message:
        return historique_messages, ""

    reponse = generer_reponse(message, historique_messages)

    historique_messages.extend([
        {"role": "user", "content": message},
        {"role": "assistant", "content": str(reponse)}
    ])

    return historique_messages, ""

with gr.Blocks(title="AstraCare") as demo:
    gr.Markdown(
        "# 🚀 AstraCare\n"
        "Pose une question sur la santé en vol spatial par écrit ou au microphone. "
        "Les réponses sont informatives et ne remplacent pas l’équipe médicale."
    )

    # Suppression du paramètre type="messages" incompatible avec cette version de Gradio
    chatbot = gr.Chatbot(label="Conversation")

    with gr.Row():
        zone_question = gr.Textbox(
            label="Ta question",
            placeholder="Ex. : Quels facteurs peuvent perturber le sommeil en mission ?",
            scale=5
        )
        bouton_envoyer = gr.Button("Envoyer", variant="primary")

    with gr.Row():
        audio = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Ou dicte ta question"
        )
        bouton_transcrire = gr.Button("Transcrire l’audio")

    bouton_transcrire.click(
        fn=transcrire_audio,
        inputs=audio,
        outputs=zone_question
    )

    bouton_envoyer.click(
        fn=envoyer_message,
        inputs=[zone_question, chatbot],
        outputs=[chatbot, zone_question]
    )

    zone_question.submit(
        fn=envoyer_message,
        inputs=[zone_question, chatbot],
        outputs=[chatbot, zone_question]
    )

demo.launch(share=False, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
def generer_reponse(question, historique=None, k=5, seuil_distance=0.65):
    question = (question or "").strip()
    if not question:
        return "Écris ou dicte une question.", []

    vecteur = modele.encode(
        [question],
        normalize_embeddings=True
    ).tolist()

    resultats = collection_rag.query(
        query_embeddings=vecteur,
        n_results=min(k, collection_rag.count()),
        include=["documents", "metadatas", "distances"]
    )

    documents = resultats["documents"][0]
    metadonnees = resultats["metadatas"][0]
    distances = resultats["distances"][0]

    extraits = []
    for doc, metadata, distance in zip(documents, metadonnees, distances):
        if not doc or distance is None or distance > seuil_distance:
            continue
        metadata = metadata or {}
        source = metadata.get("source", "Source non indiquée")
        extraits.append((source, doc))

    if not extraits:
        return (
            "Je n’ai pas trouvé de source suffisamment pertinente dans le corpus pour répondre avec confiance. "
            "Pour tout symptôme nouveau ou préoccupant, contacte l’équipe médicale de la mission.",
            []
        )

    contexte = "\n\n".join(
        f"[S{i}] Source : {source}\nExtrait : {texte[:2500]}"
        for i, (source, texte) in enumerate(extraits, start=1)
    )

    systeme = """
Tu es AstraCare, un assistant pédagogique sur la santé en vol spatial.
Réponds en français, clairement et de façon approfondie.

Appuie les faits sur les extraits fournis et cite-les avec [S1], [S2], etc.
N'invente pas de chiffres, de seuils médicaux, de diagnostics ou de traitements.
Si les extraits ne suffisent pas, indique clairement cette limite.

Pour une question sur un symptôme personnel, donne seulement des informations
générales et recommande de suivre le protocole de bord et de contacter l'équipe
médicale de la mission. En cas de symptôme grave ou d'aggravation rapide,
recommande de prévenir immédiatement l'équipe médicale et de suivre le
protocole d'urgence. Ne décide jamais si une personne est apte à poursuivre
sa mission. Tes réponses ne remplacent pas un professionnel de santé.
"""

    messages = [{"role": "system", "content": systeme}]
    for msg in (historique or [])[-8:]:
        if isinstance(msg, dict) and "role" in msg and "content" in msg:
            messages.append({"role": msg["role"], "content": msg["content"]})

    messages.append({
        "role": "user",
        "content": (
            f"Question : {question}\n\n"
            f"Extraits du corpus :\n{contexte}\n\n"
            "Réponds de manière utile et approfondie. Cite les sources dans "
            "ta réponse avec les marqueurs [S1], [S2], etc."
        )
    })

    entree = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )
    entree = {k: v.to(next(llm.parameters()).device) for k, v in entree.items()}

    with torch.inference_mode():
        sortie = llm.generate(
            **entree,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer_llm.eos_token_id
        )

    longueur_prompt = entree["input_ids"].shape[-1]
    nouveaux_tokens = sortie[0][longueur_prompt:]
    reponse = tokenizer_llm.decode(nouveaux_tokens, skip_special_tokens=True)

    sources = [f"[S{i}] {source}" for i, (source, _) in enumerate(extraits, start=1)]
    return reponse, sources